# The auto-anatomist's anneal, on a slider

Confidential, as the rest of bordercall is: keep this notebook, its copy in Colab and its outputs private.

The network learns to read the upper end-plate's width (Panjabi's EPWu) from a CT. In training a second branch also sees the vertebra's mask, and the two are mixed as (1 - w) times the CT's features plus w times the mask's. Over the run the mask goes down a ladder of three rungs as lambda rises from 0 to 1 (the vertebra's own identity goes, then its sharp edges, then the mask itself, which becomes a threshold of the CT), and then w falls from 1 to 0, so at the end the network reads the CT alone. The figure shows, every fifth step, what the mask branch is given and the width the network measures on a held-out vertebra's raw CT, beside the reference the oracle reads off that vertebra's mask.

It trains on phantoms, a demonstration and not a result. The notebook asks Colab for a T4 GPU when it opens (Runtime, Change runtime type shows what Colab gave). On the GPU it trains a network of the smoke run's widths; on a CPU, one of half those widths, which took 5.6 minutes on two CPU cores of a cloud container when this notebook was written.

## 1. Get the code

bordercall is private, and Colab's own GitHub sign-in, which opened this notebook, stays with Colab's page: the code in the notebook cannot use it. So the cell asks for your access itself, the first of these it finds:

- A token in Colab's Secrets: a fine-grained GitHub token whose resource owner is OpenSpineConsortium, with read-only access to the contents of bordercall alone, under the name `GITHUB_TOKEN` (the key on the left) with its notebook access on. The cell reads it there and never prints it.
- Else, Sign in with GitHub: the cell shows a link and a code; open the link, enter the code and authorize SymphonyBoard. The cell clones with the token GitHub grants and keeps it in no output, no file and no variable.
- Else, a ZIP: on GitHub, open bordercall at the branch that holds this notebook (main once the demo has merged), press Code, then Download ZIP (about 90 MB), and upload it when the cell asks.

The cell then installs the JAX, Flax and optax versions bordercall pins (requirements-autoanatomist.txt). On a GPU runtime it also installs the GPU build of that JAX for the CUDA version whose build Colab's image already carries (CUDA 13 on Colab's image of October 2026), so that one build alone drives the GPU, and it checks that JAX starts on the GPU before this session imports it: should it not start, the cell says why and the network trains on the CPU instead. Installing takes a minute or a few. Should Colab say a package was already imported, restart the session (Runtime, Restart session) and run the cell again: the second time it installs nothing. Run in a clone of bordercall instead of Colab, the cell fetches and installs nothing, and stops at once if the clone's Python lacks plotly, which draws the figure.

In [ ]:
import html
import http.client
import importlib.metadata
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile

REPOSITORY = "OpenSpineConsortium/bordercall"
REFS = ("main", "claude/project-thread-5hj0q5")   # the first that holds the demo is used; the second is its branch
BASE = "/content"                                  # Colab's working folder
CLONE = os.path.join(BASE, "bordercall")
PINNED = ("jax", "jaxlib", "flax", "optax")
MAJORS = ("13", "12")                              # the CUDA versions JAX's GPU builds are made for, newest first
APP_CLIENT_ID = "Iv23liTkIRov2FYN83wD"             # SymphonyBoard's, the App a reader signs in to; public, as every App's is
DEVICE_CODE = "https://github.com/login/device/code"
ACCESS_TOKEN = "https://github.com/login/oauth/access_token"


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def holds_demo(root):
    return os.path.exists(os.path.join(root, "autoanatomist", "demo.py"))


def token():
    """The GitHub token in Colab's Secrets, or None where there is none or the notebook may not read it."""
    from google.colab import userdata
    try:
        return userdata.get("GITHUB_TOKEN")
    except Exception:           # userdata's SecretNotFoundError or NotebookAccessError
        return None


def clone(secret, ref):
    """A shallow clone of ref, the token handed to git by a credential helper, so that it is in no
    command line, no remote URL and no output."""
    helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
    shutil.rmtree(CLONE, ignore_errors=True)
    subprocess.run(["git", "-c", "credential.helper=" + helper, "clone", "--quiet", "--depth", "1", "--branch", ref,
                    "https://github.com/%s.git" % REPOSITORY, CLONE], env=dict(os.environ, GITHUB_TOKEN=secret), check=True)


def unpack_upload():
    """bordercall from a ZIP of it that GitHub made and the reader uploads."""
    from google.colab import files
    print("No GITHUB_TOKEN in Colab's Secrets and no sign-in, so upload bordercall as a ZIP: on GitHub, the "
          "branch that holds this notebook, Code, Download ZIP.")
    name = next(iter(files.upload()))
    with zipfile.ZipFile(name) as archive:
        top = archive.namelist()[0].split("/")[0]
        archive.extractall(BASE)
    shutil.rmtree(CLONE, ignore_errors=True)
    os.rename(os.path.join(BASE, top), CLONE)
    if not holds_demo(CLONE):
        raise SystemExit("that ZIP holds no autoanatomist/demo.py: download the branch that holds this notebook")
    return CLONE


def json_object(answer):
    """An answer's body as a JSON object, or None where it is not one or cannot be read."""
    try:
        said = json.loads(answer.read())
    except (OSError, ValueError, http.client.HTTPException):
        return None
    return said if isinstance(said, dict) else None


def github(url, **fields):
    """One form post to GitHub's sign-in service and its answer; a refusal, an answer that is not
    JSON or a failed connection is an answer with an error, so the sign-in can say why it stopped."""
    body = urllib.parse.urlencode(dict(client_id=APP_CLIENT_ID, **fields)).encode()
    request = urllib.request.Request(url, data=body, headers={"Accept": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=30) as answer:
            said, status = json_object(answer), 200
    except urllib.error.HTTPError as refused:       # GitHub's refusals say why in JSON too
        said, status = json_object(refused), refused.code
    except (OSError, http.client.HTTPException) as failed:     # no connection, a timeout or a broken answer
        return {"error": str(failed)}
    return said if said is not None else {"error": "GitHub answered %d, not in JSON" % status}


def show_code(link, code):
    """The sign-in's page and code: a link to press where the notebook shows HTML, and words always."""
    print("Sign in with GitHub: open %s and enter the code %s" % (link, code), flush=True)
    try:
        from IPython.display import HTML, display
    except ImportError:
        return
    display(HTML('<p style="font-size:1.3em">Open <a href="%s" target="_blank">%s</a> and enter <b><code>%s</code></b></p>'
                 % (html.escape(link, quote=True), html.escape(link), html.escape(code))))


def sign_in():
    """A token from GitHub's device flow: the reader opens GitHub's page, types the code shown and
    authorizes SymphonyBoard, and the cell clones with what GitHub grants, keeping it in no output,
    no file and no variable after the clone. None where the App does not offer the flow, or the
    reader declines or lets the code expire."""
    if not APP_CLIENT_ID:
        return None
    started = github(DEVICE_CODE)
    if "device_code" not in started:
        print("GitHub's sign-in is not open to this notebook (%s)." % started.get("error_description", started.get("error")))
        return None
    show_code(started["verification_uri"], started["user_code"])
    wait, deadline = started.get("interval", 5), time.monotonic() + started.get("expires_in", 900)
    while time.monotonic() < deadline:
        time.sleep(wait)
        answer = github(ACCESS_TOKEN, device_code=started["device_code"],
                        grant_type="urn:ietf:params:oauth:grant-type:device_code")
        if "access_token" in answer:
            print("Signed in.")
            return answer["access_token"]
        if answer.get("error") == "slow_down":
            wait = answer.get("interval", wait + 5)
        elif answer.get("error") != "authorization_pending":
            print("GitHub's sign-in stopped (%s)." % answer.get("error_description", answer.get("error")))
            return None
    print("The sign-in code expired.")
    return None


def fetch():
    """bordercall at the first of REFS that holds the demo, once per session: with the token in Colab's
    Secrets, else one from signing in with GitHub, else from a ZIP uploaded by hand."""
    if holds_demo(CLONE):
        return CLONE
    secret = token() or sign_in()
    if secret is None:
        return unpack_upload()
    for ref in REFS:
        clone(secret, ref)
        if holds_demo(CLONE):
            return CLONE
    raise SystemExit("none of %s holds autoanatomist/demo.py" % ", ".join(REFS))


def repository_root():
    """Outside Colab: the clone this notebook sits in, the first folder up that holds autoanatomist/."""
    here = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(here, "autoanatomist")):
        if os.path.dirname(here) == here:
            raise SystemExit("run this notebook in a clone of bordercall, or in Colab")
        here = os.path.dirname(here)
    return here


def has_gpu():
    """Whether this session has an NVIDIA GPU, as Colab's GPU runtimes do."""
    try:
        listed = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    except OSError:
        return False
    return listed.startswith("GPU")


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def plugins(major):
    """JAX's GPU build for one CUDA version: its plugin and the plugin's PJRT library."""
    return ("jax-cuda%s-plugin" % major, "jax-cuda%s-pjrt" % major)


def driver_major():
    """The newest CUDA version the GPU's driver runs, from nvidia-smi's header ("CUDA Version: 13.0"),
    or None where nvidia-smi says nothing of it."""
    try:
        header = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    except OSError:
        return None
    _, said, rest = header.partition("CUDA Version:")
    major = rest.strip().split(".")[0]
    return major if said and major.isdigit() else None


def cuda_major():
    """The CUDA version of the GPU build to install: the one whose build the image carries already
    (13 on Colab's GPU image of October 2026), so that pip replaces it and leaves no second plugin
    to take the GPU; else the newest the driver runs; else 12."""
    for major in MAJORS:
        if any(installed(name) for name in plugins(major)):
            return major
    driver = driver_major()
    for major in MAJORS:
        if driver is not None and int(driver) >= int(major):
            return major
    return "12"


def stale_plugins(major):
    """The image's GPU builds for the other CUDA versions: left beside the one installed, either
    could take the GPU, so pip takes them out."""
    return [name for other in MAJORS if other != major for name in plugins(other) if installed(name)]


def pins(root, cuda=None):
    """jax, jaxlib, flax and optax as bordercall pins them, and for a CUDA version the GPU build of
    that jaxlib, which JAX releases under jaxlib's version."""
    found = {}
    with open(os.path.join(root, "requirements-autoanatomist.txt")) as f:
        for line in f:
            name, _, version = line.split(";")[0].strip().partition("==")
            if name in PINNED and version:
                found[name] = version
    if cuda:
        found.update({name: found["jaxlib"] for name in plugins(cuda)})
    return found


def spec(name, version):
    """pip's words for one pin; a GPU plugin's extra brings NVIDIA's libraries, which pip leaves as
    they are where the image's meet it."""
    extra = "[with-cuda]" if name.startswith("jax-cuda") and name.endswith("-plugin") else ""
    return "%s%s==%s" % (name, extra, version)


def install(wanted, stale=()):
    """pip takes out the stale plugins and installs the pins the session lacks; where that replaces
    a package this session has imported already, numpy most often, the session must restart before
    it can import the new one."""
    missing = [spec(name, version) for name, version in wanted.items() if installed(name) != version]
    stale = [name for name in stale if installed(name)]
    if not missing and not stale:
        return
    if missing and sys.version_info < (3, 12):
        raise SystemExit("bordercall's pinned JAX needs Python 3.12 or newer; this session has %d.%d" % sys.version_info[:2])
    owners = importlib.metadata.packages_distributions()
    imported = sorted({d for m in list(sys.modules) for d in owners.get(m.split(".")[0], [])})
    before = {d: installed(d) for d in imported}
    if stale:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "--yes", "--quiet", *stale], check=True)
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
    replaced = [d for d in imported if installed(d) != before[d]]
    if replaced:
        raise SystemExit("pip replaced %s, which this session had imported already: restart the session "
                         "(Runtime, Restart session) and run this cell again" % ", ".join(replaced))


def ensure_plotly():
    """plotly draws the figure. Colab has it; elsewhere it is the reader's to install, since bordercall's
    requirements do not pin it, and the cell says so before the minutes of training rather than after."""
    if importlib.util.find_spec("plotly") is not None:
        return
    if not in_colab():
        raise SystemExit("the figure needs plotly, which bordercall's requirements do not pin: "
                         "pip install plotly, then run this cell again")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "plotly"], check=True)


def why_no_gpu():
    """None where JAX starts on the GPU in a child process; else why not, in the child's last words.
    A GPU that JAX registers and then fails to start makes it raise at the first use, so this
    session asks a child first and imports JAX itself only after."""
    env = dict(os.environ, XLA_PYTHON_CLIENT_PREALLOCATE="false")
    env.pop("JAX_PLATFORMS", None)
    child = subprocess.run([sys.executable, "-c", "import jax; print(jax.default_backend())"], env=env,
                           capture_output=True, text=True)
    backend = child.stdout.split()[-1:]
    if child.returncode == 0 and backend == ["gpu"]:
        return None
    lines = [line.strip() for line in child.stderr.splitlines() if line.strip()]
    said = [line for line in lines if "Error" in line or "Warning" in line]
    return (said or lines or ["JAX chose its %s backend" % (backend or ["no"])[0]])[-1]


def settle_platform(cuda):
    """None where JAX will run on the GPU; else why not, with JAX_PLATFORMS set to the CPU before
    this session imports JAX."""
    why = why_no_gpu() if cuda else "this session has no GPU"
    if why:
        os.environ["JAX_PLATFORMS"] = "cpu"
    return why


def prepare():
    """bordercall's folder, the CUDA version of the GPU build in use (None without a GPU) and why JAX
    will not run on the GPU (None where it will), each step where it must come: the pins and the GPU
    build installed with the other builds taken out first, plotly asked for before the minutes of
    training, and JAX_PLATFORMS settled before this session imports JAX, which reads it only then."""
    root = fetch() if in_colab() else repository_root()
    cuda = cuda_major() if has_gpu() else None
    if in_colab():
        install(pins(root, cuda), stale_plugins(cuda) if cuda else ())
    ensure_plotly()
    return root, cuda, settle_platform(cuda)


ROOT, CUDA, WHY_CPU = prepare()
sys.path.insert(0, ROOT)

import jax  # noqa: E402
from autoanatomist import demo  # noqa: E402

where = "an uploaded ZIP"
if os.path.exists(os.path.join(ROOT, ".git")):       # a folder in a clone, a file in a worktree
    where = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print("bordercall at", where, "| jax", jax.__version__, "on", jax.devices()[0])
if CUDA and WHY_CPU:
    print("This session has a GPU, but JAX did not start on it (%s), so it trains on the CPU, with the "
          "smaller network." % WHY_CPU)

## 2. Train, with a snapshot every fifth step

The smoke run's phantoms (8 short columns of three vertebrae to train on, 4 held out), its schedule, seeds and 150 steps, on the one measurement, with the naming layer off; the smoke run itself trains all seven measurements with the naming layer on. Every 25 steps it prints the width's mean error over the 12 held-out vertebrae. `WIDTHS` sets the network's size: on a GPU, (16, 32, 64), the smoke run's widths; on the CPU, (8, 16, 32). On two CPU cores of a cloud container with nothing else running, the whole run, making the phantoms included, took 5.6 minutes at (8, 16, 32) and 8.6 at (16, 32, 64); Colab's own CPUs and its T4 are not timed yet.

In [ ]:
from autoanatomist.model import Config

WIDTHS = (16, 32, 64) if jax.default_backend() == "gpu" else (8, 16, 32)
run = demo.run(cfg=Config(keys=(demo.KEY,), widths=WIDTHS))
print("%d snapshots in %.0f s" % (len(run.snapshots), run.seconds))

## 3. The figure

How to read it:

- Drag the slider under the figure, or press Play: each position is the network after that many steps.
- Left: one held-out vertebra's CT drawn as its bone surface at 200 HU, with no mask anywhere: the one whose error at the end is the median of the 12, unless `VERTEBRA` names another. Orange is the end-plate width the network measures from the CT alone, blue the reference the oracle read off the mask. Drag to turn it, scroll to zoom.
- Top right: what the mask branch is given at that step, in green over a training vertebra's CT, drawn at its weight w. It starts as the vertebra's own mask, takes in its neighbours, blurs and turns into a threshold of the CT, then fades out as w falls to 0, leaving the CT alone.
- Middle right: lambda (black) and w (green) over the run, the dots at the slider's step.
- Bottom right: the width on that vertebra over the run, the network's (orange) against the reference (blue, dashed). The title gives both, and the mean error over all 12 held-out vertebrae.

In [ ]:
VERTEBRA = run.typical()   # the one whose error at the end is the median of the 12; or any of 0 to 11
print("vertebra", VERTEBRA, "of", run.levels)
demo.figure(run, vertebra=VERTEBRA).show()

## What it is not

- Phantoms, not patients: each column is drawn from Panjabi's tables with 20 HU of noise, and a few minutes of training on 24 of their vertebrae show the anneal, not how well it reads a real CT. Nothing is kept.
- The network reads the CT alone, but its crop is placed as in training, from the oracle's frame turned and shifted at random. Placing the crop from the CT alone (scanimport's mask-free mode, `crops.from_found`) is not measured yet.
- The reference is the stand-in oracle's, which reads the phantoms' drawn values back within 0.44 mm on average and 1.34 mm at most (autoanatomist/README.md).